# Meshek-ML: Perishable Inventory Optimization — Quickstart

This notebook walks through the full pipeline:
1. Generate synthetic demand data for Israeli greengrocers
2. Explore demand patterns, seasonality, and spoilage
3. Train a PPO reinforcement learning agent to optimize ordering
4. Compare the RL agent against a classical newsvendor baseline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/meshek-ml/blob/main/notebooks/colab_quickstart.ipynb)

**This notebook requires a fresh Colab runtime. Run cells 1–4 in order to set up the environment.**

## 1. Environment Setup

Install the package and its dependencies from the repository's `pyproject.toml`.
This pulls in simulation, forecasting, optimization, and experiment tracking extras.

In [ ]:
import subprocess
import sys

# Clone the repository (idempotent — skips if already cloned)
!git clone https://github.com/YOUR_USERNAME/meshek-ml.git 2>/dev/null || echo "Repository already cloned."
%cd meshek-ml

# Install the package with simulation, forecasting, and optimization extras
# This reads dependency groups from pyproject.toml
!pip install -q -e ".[simulation,forecasting,optimization,tracking]"

# Verify installation
import meshek_ml
print(f"meshek-ml installed successfully")

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU available: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("No GPU detected — CPU mode. PPO training will be slower but functional.")
    print("Tip: Runtime > Change runtime type > GPU to enable GPU acceleration.")

## 2. Google Drive Access

Mount Google Drive to read real sales data and save outputs persistently across sessions.
If running locally (not in Colab), this falls back to local directories.

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')

    # Create a project output directory on Drive
    DRIVE_OUTPUT = "/content/drive/MyDrive/meshek-ml-outputs"
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)
    print(f"Drive mounted. Output directory: {DRIVE_OUTPUT}")

    # Optional: path to real data (user sets this)
    DRIVE_DATA_INPUT = "/content/drive/MyDrive/meshek-ml-data"
    if os.path.exists(DRIVE_DATA_INPUT):
        print(f"Data input directory found: {DRIVE_DATA_INPUT}")
        print(f"Files: {os.listdir(DRIVE_DATA_INPUT)}")
    else:
        print(f"No data input directory at {DRIVE_DATA_INPUT} — using synthetic data only.")
except ImportError:
    print("Not running in Google Colab — skipping Drive mount.")
    DRIVE_OUTPUT = "./outputs"
    DRIVE_DATA_INPUT = "./data"
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)
    print(f"Local output directory: {DRIVE_OUTPUT}")

## 3. Generate Synthetic Data

We simulate 3 merchants over 6 months. Each merchant has a different archetype
(small urban boutique, suburban shop, market stall, etc.) and sells 8 types of
produce with realistic demand patterns.

In [ ]:
from meshek_ml.simulation.generator import run_simulation

df = run_simulation(
    n_merchants=3,
    start_date="2024-01-01",
    end_date="2024-06-30",
    seed=42,
)

print(f"Dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Merchants: {df['merchant_id'].nunique()}")
print(f"Products: {df['product'].nunique()}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
df.head()

## 4. Explore the Data

In [ ]:
import matplotlib.pyplot as plt

# Pick one merchant to visualize
merchant_id = df["merchant_id"].unique()[0]
merchant_df = df[df["merchant_id"] == merchant_id].copy()

fig, ax = plt.subplots(figsize=(14, 5))
for product, group in merchant_df.groupby("product"):
    ax.plot(group["date"], group["realized_demand"], label=product, alpha=0.7)

ax.set_title(f"Daily Demand — Merchant: {merchant_id}")
ax.set_xlabel("Date")
ax.set_ylabel("Units demanded")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Weekly demand pattern (averaged across all merchants/products)
df["day_of_week"] = df["date"].dt.day_name()
day_order = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
weekly = df.groupby("day_of_week")["realized_demand"].mean().reindex(day_order)

fig, ax = plt.subplots(figsize=(8, 4))
weekly.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Average Demand by Day of Week")
ax.set_ylabel("Mean demand")
ax.set_xticklabels(day_order, rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from meshek_ml.simulation.spoilage import weibull_quality

# Spoilage curves for different product types
days = np.arange(0, 15)
products_spoilage = {
    "Strawberries (shelf=3d)": {"shape": 2.0, "scale": 3.0},
    "Tomatoes (shelf=6d)": {"shape": 2.0, "scale": 6.0},
    "Oranges (shelf=14d)": {"shape": 2.0, "scale": 14.0},
    "Apples (shelf=30d)": {"shape": 2.0, "scale": 30.0},
}

fig, ax = plt.subplots(figsize=(10, 5))
for label, params in products_spoilage.items():
    quality = weibull_quality(days, **params)
    ax.plot(days, quality, label=label, linewidth=2)

ax.axhline(y=0.3, color="red", linestyle="--", alpha=0.5, label="Spoilage threshold")
ax.set_title("Product Quality Decay (Weibull Model)")
ax.set_xlabel("Age (days)")
ax.set_ylabel("Quality (0–1)")
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 5. Train an RL Agent (PPO)

We'll train a PPO agent to decide **how much to order each day** for a single
product. The agent learns to balance:
- Revenue from sales
- Waste from spoiled items
- Stockouts (lost sales)
- Holding costs

In [ ]:
from meshek_ml.optimization.env import PerishableInventoryEnv
from meshek_ml.optimization.rewards import CostParams

# Create environment — simulates a product like Tomatoes
costs = CostParams(
    selling_price=7.0,
    purchase_cost=5.0,
    holding_cost=0.1,
    waste_penalty=5.0,
    stockout_penalty=3.0,
)

env = PerishableInventoryEnv(
    max_shelf_life=7,
    max_order=50,
    demand_mean=20.0,
    demand_dispersion=3.0,
    episode_length=90,
    costs=costs,
)

print(f"Observation space: {env.observation_space.shape}")
print(f"Action space: {env.action_space}")

In [ ]:
from meshek_ml.optimization.ppo_agent import train_ppo

# Train for 50k steps (~2 min on Colab GPU, ~5 min on CPU)
model = train_ppo(
    env,
    total_timesteps=50_000,
    learning_rate=3e-4,
    n_steps=2048,
)
print("Training complete!")

## 6. Evaluate: PPO vs Newsvendor Baseline

The **newsvendor** is a classical operations research model — it computes the
optimal fixed order quantity given demand distribution and cost parameters.
Let's see if the RL agent can beat it by adapting to inventory state.

In [ ]:
from meshek_ml.optimization.newsvendor import optimal_order_negbin
from meshek_ml.optimization.evaluation import compute_inventory_metrics


def run_episode(env, policy_fn, seed=0):
    """Run one episode and collect metrics."""
    obs, _ = env.reset(seed=seed)
    total_demand = 0
    total_sold = 0
    total_wasted = 0
    total_ordered = 0
    stockout_days = 0
    rewards = []
    history = []  # for plotting

    done = False
    while not done:
        action = policy_fn(obs)
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        total_demand += info["demand"]
        total_sold += info["sold"]
        total_wasted += info["wasted"]
        total_ordered += int(action[0])
        if info["unmet_demand"] > 0:
            stockout_days += 1
        rewards.append(reward)
        history.append(info | {"reward": reward, "order": int(action[0])})

    metrics = compute_inventory_metrics(
        total_demand=total_demand,
        total_sold=total_sold,
        total_wasted=total_wasted,
        total_ordered=total_ordered,
        total_stockout_events=stockout_days,
        n_days=env.episode_length,
    )
    metrics["total_reward"] = sum(rewards)
    return metrics, history


# PPO policy
def ppo_policy(obs):
    action, _ = model.predict(obs, deterministic=True)
    return action


# Newsvendor: fixed order quantity each day
nv_qty = optimal_order_negbin(
    mean_demand=20.0,
    dispersion=3.0,
    underage_cost=costs.stockout_penalty,
    overage_cost=costs.waste_penalty,
)
print(f"Newsvendor optimal order: {nv_qty} units/day")


def newsvendor_policy(obs):
    return np.array([nv_qty], dtype=np.float32)


# Run evaluation episodes
n_eval = 10
ppo_results = []
nv_results = []
for i in range(n_eval):
    ppo_m, ppo_h = run_episode(env, ppo_policy, seed=1000 + i)
    nv_m, nv_h = run_episode(env, newsvendor_policy, seed=1000 + i)
    ppo_results.append(ppo_m)
    nv_results.append(nv_m)

In [ ]:
import pandas as pd

# Compare metrics
ppo_df = pd.DataFrame(ppo_results).mean()
nv_df = pd.DataFrame(nv_results).mean()

comparison = pd.DataFrame({"PPO Agent": ppo_df, "Newsvendor": nv_df})
comparison.index.name = "Metric"

# Format for display
display(comparison.style.format("{:.2f}").set_caption(
    f"Average over {n_eval} evaluation episodes (90 days each)"
))

## 7. Visualize a Single Episode

In [ ]:
# Run one detailed episode for each policy
_, ppo_history = run_episode(env, ppo_policy, seed=42)
_, nv_history = run_episode(env, newsvendor_policy, seed=42)

ppo_hist = pd.DataFrame(ppo_history)
nv_hist = pd.DataFrame(nv_history)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Inventory levels
axes[0].plot(ppo_hist["stock"], label="PPO", alpha=0.8)
axes[0].plot(nv_hist["stock"], label="Newsvendor", alpha=0.8)
axes[0].set_ylabel("Stock on hand")
axes[0].set_title("Inventory Levels Over 90 Days")
axes[0].legend()

# Waste
axes[1].bar(range(len(ppo_hist)), ppo_hist["wasted"], alpha=0.6, label="PPO", width=0.4)
axes[1].bar(
    [x + 0.4 for x in range(len(nv_hist))],
    nv_hist["wasted"],
    alpha=0.6,
    label="Newsvendor",
    width=0.4,
)
axes[1].set_ylabel("Units wasted")
axes[1].set_title("Daily Waste")
axes[1].legend()

# Cumulative reward
axes[2].plot(ppo_hist["reward"].cumsum(), label="PPO", alpha=0.8)
axes[2].plot(nv_hist["reward"].cumsum(), label="Newsvendor", alpha=0.8)
axes[2].set_ylabel("Cumulative reward")
axes[2].set_xlabel("Day")
axes[2].set_title("Cumulative Profit")
axes[2].legend()

plt.tight_layout()
plt.show()

## Next Steps

- **More training**: Increase `total_timesteps` to 200k+ for better convergence
- **Different products**: Try strawberries (shelf_life=3, harder!) or apples (shelf_life=30, easier)
- **Demand forecasting**: Use `meshek_ml.forecasting` to predict demand and feed forecasts into the ordering policy
- **Federated learning**: Train models across multiple merchants without sharing raw data — see `meshek_ml.federated`